# 02 — Petrobras 3W: challenge the sector-agnostic contract

**Outcome:** make a second, deliberately different sector pass through the
same neutral `SPEC-CORE`/`SPEC-EVAL` shapes while recording what cannot be
represented without invention.

This is a contract test, not an oil-production realism project. It uses
the smallest deterministic subset of **real** 3W wells that jointly covers:
normal data, a transient event, a persistent condition, a state
transition, a missing/frozen measurement, and at least three distinct
wells.

## 1. Setup and Drive paths

Upload the unpacked public source to:

`MyDrive/anomaly_detection/sources/petrobras_3w/2.0.0/raw/3w_dataset_2.0.0/`

Keep `dataset.ini`, `README.md`, `LICENSE-CC-BY`, and directories `0`–`9`.

In [ ]:
import json
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")

DRIVE_ROOT = Path(os.getenv(
    "ANOMALY_DRIVE_ROOT",
    "/content/drive/MyDrive/anomaly_detection",
))
NOTEBOOK_HOME = Path(os.getenv(
    "ANOMALY_NOTEBOOK_HOME",
    DRIVE_ROOT / "research" / "week1",
))
if str(NOTEBOOK_HOME) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_HOME))

from week1_core import (
    CORE_TABLES,
    CORE_VERSION,
    discover_threew,
    materialise_threew,
    select_threew_subset,
    write_json,
)

THREEW_SOURCE = Path(os.getenv(
    "THREEW_SOURCE_ROOT",
    str(
        DRIVE_ROOT / "sources" / "petrobras_3w" / "2.0.0"
        / "raw" / "3w_dataset_2.0.0"
    ),
))
RUN_ID = os.getenv("THREEW_RUN_ID", "contract_challenge_v1")
RUN_ROOT = (
    DRIVE_ROOT / "outputs" / "research" / f"v{CORE_VERSION}"
    / "petrobras_3w" / RUN_ID
)
COPY_PINNED_FIXTURE = os.getenv("COPY_PINNED_FIXTURE", "1") == "1"
FIXTURE_ROOT = (
    DRIVE_ROOT / "fixtures" / "petrobras_3w" / "2.0.0"
    / f"v{CORE_VERSION}" / RUN_ID
)
RUN_MATERIALISATION = os.getenv("RUN_MATERIALISATION", "1") == "1"

display(pd.Series({
    "source": str(THREEW_SOURCE),
    "run_root": str(RUN_ROOT),
    "fixture_copy": str(FIXTURE_ROOT) if COPY_PINNED_FIXTURE else "disabled",
}, name="value").to_frame())

## 2. OilWell Pack — only what the source supports

The public 3W source provides 27 measurements plus `class` and `state`
labels. The pack maps those measurements into neutral kinds and units.

We do **not** invent manifold topology, shared-cause groups, tickets,
maintenance events, customer impact, or severity. Empty relations and
events are valid contract states. `class` and `state` never enter
`SPEC-CORE`.

In [ ]:
STATE_FIELDS = [
    "ESTADO-DHSV", "ESTADO-M1", "ESTADO-M2", "ESTADO-PXO",
    "ESTADO-SDV-GL", "ESTADO-SDV-P", "ESTADO-W1", "ESTADO-W2",
    "ESTADO-XO",
]
OPENING_FIELDS = ["ABER-CKGL", "ABER-CKP"]
PRESSURE_FIELDS = [
    "P-ANULAR", "P-JUS-BS", "P-JUS-CKGL", "P-JUS-CKP",
    "P-MON-CKGL", "P-MON-CKP", "P-MON-SDV-P", "P-PDG",
    "PT-P", "P-TPT",
]
FLOW_FIELDS = ["QBS", "QGL"]
TEMPERATURE_FIELDS = ["T-JUS-CKP", "T-MON-CKP", "T-PDG", "T-TPT"]

def oil_metric(native_field, family, kind, unit, direction):
    return {
        "native_field": native_field,
        "metric_id": f"oil_well.{family}.{native_field.lower().replace('-', '_')}",
        "entity_type": "oil_well",
        "measurement_kind": kind,
        "unit": unit,
        "anomaly_direction": direction,
        "sampling_semantics": "value_at_one_second_sample",
        "aggregation_semantics": (
            "last" if kind == "discrete_state" else "median"
        ),
        "value_nullable": True,
        "expected_cadence_seconds": 1,
        "censoring_type": "source_null_only",
        "expected_behaviour_profile": (
            "state_transition" if kind == "discrete_state"
            else "stable_continuous"
        ),
        "lower_bound": None,
        "upper_bound": None,
        "candidate_periods": [],
        "exposure_metric_id": None,
        "exposure_semantics": None,
        "exposure_unit": None,
        "exposure_formula_id": None,
        "exposure_source": None,
        "context_keys": [],
    }

metric_catalogue = pd.DataFrame(
    [oil_metric(field, "state", "discrete_state", "state_code", "change")
     for field in STATE_FIELDS]
    + [oil_metric(field, "opening", "gauge", "percent", "both")
       for field in OPENING_FIELDS]
    + [oil_metric(field, "pressure", "gauge", "source_pressure_unit", "both")
       for field in PRESSURE_FIELDS]
    + [oil_metric(field, "flow", "gauge", "source_flow_unit", "both")
       for field in FLOW_FIELDS]
    + [oil_metric(field, "temperature", "gauge", "source_temperature_unit", "both")
       for field in TEMPERATURE_FIELDS]
)

assert len(metric_catalogue) == 27
assert metric_catalogue["metric_id"].is_unique
assert {"class", "state"}.isdisjoint(metric_catalogue["native_field"])
display(metric_catalogue)

## 3. Pin the smallest real-well challenge subset

Selection is deterministic and hash-pinned. The count is not fixed in
advance: it is the smallest subset satisfying every criterion, so no
criterion is quietly weakened to force “three to five files.”

In [ ]:
inventory = discover_threew(THREEW_SOURCE)
display(pd.Series(inventory, name="value").to_frame())
assert inventory["ready"], inventory

selected = select_threew_subset(
    THREEW_SOURCE, metric_catalogue["native_field"]
)
selection_table = pd.DataFrame([
    {
        "file": item.relative_path,
        "entity_id": item.entity_id,
        "event_code": item.event_code,
        "rows": item.rows,
        "coverage": ", ".join(sorted(item.coverage)),
    }
    for item in selected
])
required_coverage = {
    "normal_instance", "transient_event", "persistent_condition",
    "state_transition", "missing_or_frozen_measurement",
}
covered = set().union(*(item.coverage for item in selected))
assert required_coverage <= covered
assert len({item.entity_id for item in selected}) >= 3
assert all(item.source_kind == "real" for item in selected)
display(selection_table)

## 4. Translate through the same neutral contract

Physical batches are fixed at 5,000 native rows. The canonical long table
is much larger than the wide source, but no full 8-million-row canonical
panel is held in memory.

In [ ]:
if RUN_MATERIALISATION:
    workflow_report = materialise_threew(
        THREEW_SOURCE,
        RUN_ROOT,
        metric_catalogue,
        fixture_destination=FIXTURE_ROOT if COPY_PINNED_FIXTURE else None,
    )
else:
    workflow_report = json.loads(
        (RUN_ROOT / "workflow_report.json").read_text()
    )

core = RUN_ROOT / "SPEC-CORE"
evaluation = RUN_ROOT / "SPEC-EVAL"
core_manifest = json.loads((core / "manifest.json").read_text())
eval_manifest = json.loads((evaluation / "manifest.json").read_text())
display(pd.Series(core_manifest["row_counts"], name="rows").to_frame())
display(pd.Series(eval_manifest["row_counts"], name="rows").to_frame())

## 5. Cross-sector assertions

Passing means the generic tables accept another sector without a
telecom-specific branch. It does **not** mean every oil-well concept is
represented. The contract-fit report below records omissions and
distortions explicitly.

In [ ]:
relations = pd.read_parquet(core / "entity_relations.parquet")
events = pd.read_parquet(core / "operational_events.parquet")
registry = pd.read_parquet(core / "entity_registry.parquet")
conditions = pd.read_parquet(evaluation / "gt_condition_states.parquet")
causes = pd.read_parquet(evaluation / "gt_cause_groups.parquet")
on_disk_catalogue = pd.read_parquet(core / "metric_catalogue.parquet")

assert set(CORE_TABLES) == set(core_manifest["row_counts"])
assert core_manifest["truth_columns_removed"] == ["class", "state"]
assert "class" not in on_disk_catalogue.columns
assert "state" not in on_disk_catalogue.columns
assert relations.empty
assert events.empty
assert len(registry) >= 3
assert len(conditions) > 0
assert "severity_ordinal" not in conditions.columns
assert causes.empty
assert not (evaluation / "gt_ticket_links.parquet").exists()
assert core_manifest["physical_telemetry_batch_native_rows"] == 5_000

print("PASS — a second sector uses the neutral contract without invented facts")

## 6. Contract-fit report: what worked, what did not

This report is the actual scientific result of the challenge. A sector
adapter can appear successful by pushing every awkward concept into a
pack; therefore we explicitly record what could not be expressed and what
was distorted.

In [ ]:
contract_fit_report = {
    "contract_version": CORE_VERSION,
    "source": "Petrobras 3W Dataset 2.0.0",
    "represented_cleanly": [
        "timestamped multivariate telemetry",
        "oil-well entity identity and observed validity interval",
        "null-valued measurements retained as invalid observations",
        "event labels isolated in gt_fault_events and entity intervals",
        "graded source state represented as gt_condition_states without invented severity",
        "empty relations and operational events represented honestly",
    ],
    "not_expressible_without_invention": [
        "physical manifold or shared-infrastructure topology",
        "shared causal groups between wells",
        "operator tickets and maintenance actions",
        "customer or production impact severity",
        "fault resolution after a source instance ends",
    ],
    "distortions_or_limitations": [
        "source pressure, flow, and temperature units are not declared in the Parquet schema; pack units remain explicitly source-defined",
        "each source file is an experimental instance, while entity_id is inferred from the real-well filename",
        "state labels are intervals, not onset/impact/resolution events",
        "no non-tree relation is exercised by 3W because topology is absent, not because the contract requires a tree",
    ],
    "generic_contract_change_required": [
        "retain gt_condition_states alongside event-shaped truth"
    ],
    "deliberately_not_added": [
        "severity_ordinal", "gt_terminal_outcomes",
        "invented manifold topology", "invented cause groups",
    ],
}
report_path = RUN_ROOT / "contract_fit_report.json"
if not report_path.exists():
    write_json(report_path, contract_fit_report)
display(pd.Series(contract_fit_report, name="finding").to_frame())
print("Saved:", report_path)
print("NEXT: run Notebook 03 to produce a ranked incident list.")